# 1. Setup Environment and Imports


In [ ]:
# !pip install -q torch pandas matplotlib huggingface_hub

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt


# 2. Define SAE Architecture Classes


In [ ]:
class BatchTopKSAE(nn.Module):
    def __init__(self, d_in, d_sae, k):
        super().__init__()
        self.k       = k
        self.b_pre   = nn.Parameter(torch.zeros(d_in))
        self.encoder = nn.Linear(d_in, d_sae, bias=True)
        self.decoder = nn.Linear(d_sae, d_in, bias=True)

    def encode(self, x):
        x_centered = x - self.b_pre
        pre_acts   = self.encoder(x_centered)
        n_keep     = int(pre_acts.numel() * self.k / pre_acts.shape[-1])
        threshold  = pre_acts.reshape(-1).topk(n_keep).values.min()
        acts       = pre_acts * (pre_acts >= threshold).float()
        return F.relu(acts)

    def decode(self, z):
        return self.decoder(z) + self.b_pre

    def forward(self, x):
        z     = self.encode(x)
        recon = self.decode(z)


# 3. Download and Load Grid Search Results


In [ ]:
csv_path = hf_hub_download(
    repo_id="<namespace>/sae-grid-search-layer12",
    filename="results.csv",
    repo_type="model"
)

results_df = pd.read_csv(csv_path)


# 4. Visualize Hyperparameter Sweep


In [ ]:
import seaborn as sns

if 'Dead %' in results_df.columns and results_df['Dead %'].dtype == object:
    results_df['Dead %'] = results_df['Dead %'].str.rstrip('%').astype('float') / 100.0

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

sns.lineplot(data=results_df, x='K', y='Val MSE', hue='Expansion', style='Type', markers=True, ax=axes[0])
axes[0].set_title('Val MSE vs K')

sns.lineplot(data=results_df, x='K', y='NMSE', hue='Expansion', style='Type', markers=True, ax=axes[1])
axes[1].set_title('NMSE vs K')

sns.lineplot(data=results_df, x='K', y='L0', hue='Expansion', style='Type', markers=True, ax=axes[2])
axes[2].set_title('L0 vs K')

sns.lineplot(data=results_df, x='K', y='Dead %', hue='Expansion', style='Type', markers=True, ax=axes[3])
axes[3].set_title('Dead Feature % vs K')

plt.tight_layout()


# 5. Load Specific SAE Checkpoint


In [ ]:
path = hf_hub_download(
    repo_id="<namespace>/sae-grid-search-layer12",
    filename="BatchTopK_exp8x_k220.pt",  # change as needed
    repo_type="model"
)

sae = BatchTopKSAE(d_in=896, d_sae=7168, k=220)
sae.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
sae.eval()


# 6. Evaluate SAE Forward Pass


In [ ]:
# Create dummy activation data representing base activations (instruct_base layer 12)
N = 512
d_in = 896
x_dummy = torch.randn(N, d_in)

with torch.no_grad():
    x_hat, z = sae(x_dummy)

print(f"Input shape: {x_dummy.shape}")
print(f"Reconstructed shape: {x_hat.shape}")
print(f"Latent shape: {z.shape}")

l0_norm = (z > 0).float().sum(dim=-1).mean().item()


# 11. Representation Geometry (Linear CKA)


# 12. Weight Norm Distributions (Sparsity Proxies)


# 13. Polysemanticity Surrogate (Subspace Variance)
